In [0]:
%pip install openpyxl
%pip install pymc arviz

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import pymc as pm
import arviz as az

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"PyMC  : {pm.__version__}")
print(f"ArviZ : {az.__version__}")

# Data Ingestion

In [0]:
file_path = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/EXH_ModelData.xlsx"

df = pd.read_excel(file_path, sheet_name=0)
df = df.sort_values("week_date").reset_index(drop=True)

print(f"Shape : {df.shape}")
print(f"Range : {df['week_date'].min().date()} → {df['week_date'].max().date()}")
print(f"Weeks : {len(df)}")
print(f"Missing values: {df.isna().sum().sum()}")

In [0]:
date_col     = "week_date"
target_col   = "total_weekly_attendance"
control_cols = df.columns[3:45].tolist()

assert target_col not in control_cols
assert date_col not in control_cols

week_diffs = df[date_col].diff().dropna().dt.days.value_counts()
print(f"Date     : {date_col}")
print(f"Target   : {target_col}")
print(f"Controls : {len(control_cols)} columns")
print(f"Week spacing: {week_diffs.to_dict()}")

In [0]:
df["stat_holiday_in_week"] = (df["stat_holiday_in_week"] == "Y").astype(int)
df["is_long_weekend"]      = (df["is_long_weekend"] == "Y").astype(int)

season_dummies = pd.get_dummies(df["season"], prefix="season", drop_first=False).astype(int)
season_dummies = season_dummies.drop(columns=["season_winter"])
df = pd.concat([df.drop(columns=["season"]), season_dummies], axis=1)

control_cols = [c for c in control_cols if c != "season"] + season_dummies.columns.tolist()

non_numeric = df[control_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Final control count : {len(control_cols)}")
print(f"Non-numeric remaining: {non_numeric if non_numeric else 'None ✓'}")

# Feature Engineering

In [0]:
# --- Correlation heatmap of ORIGINAL 45 features (before any drops) ---
n = len(df)
n_test  = 15
n_val   = 15
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy()
val_df   = df.iloc[n_train : n_train + n_val].copy()
test_df  = df.iloc[n_train + n_val :].copy()

# Rebuild the full feature list to show what we started with
full_control_cols = control_cols.copy()  # this has all 45 after encoding

corr_matrix = train_df[full_control_cols + [target_col]].corr()

# Build a custom palette: red = negative, white = none, blue = positive
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    corr_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    annot=False,
    square=True,
    cbar_kws={"shrink": 0.6, "label": "Pearson correlation"},
    linewidths=0.3,
    linecolor="white",
    ax=ax,
)
ax.set_title(f"Feature correlation matrix — all {len(full_control_cols)} controls + target",
             fontsize=14, pad=15)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
# plt.savefig("/dbfs/FileStore/mmm/report_corr_heatmap_full.png", dpi=120, bbox_inches="tight")
plt.show()

# Quantify the multicollinearity story
print("\n" + "═" * 65)
print("MULTICOLLINEARITY EVIDENCE")
print("═" * 65)

# Pairs with very high correlation (the smoking gun)
high_corr_pairs = []
for i, c1 in enumerate(full_control_cols):
    for c2 in full_control_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.7:
            high_corr_pairs.append((c1, c2, r))

high_corr_pairs.sort(key=lambda x: -abs(x[2]))
print(f"\nFeature pairs with |correlation| > 0.7: {len(high_corr_pairs)}")
print(f"{'Feature 1':<30} {'Feature 2':<30} {'Correlation':>12}")
print("─" * 75)
for c1, c2, r in high_corr_pairs[:20]:
    print(f"{c1:<30} {c2:<30} {r:>+12.3f}")

# Correlations with target — which features actually carry signal?
target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
print(f"\n\nTop 15 features by |correlation with attendance|:")
print(f"{'Feature':<32} {'r':>8}")
print("─" * 42)
for feat, r in target_corr.head(15).items():
    print(f"{feat:<32} {r:>+8.3f}")

print(f"\n\nFeatures with |r| < 0.10 with attendance (weak signal):")
weak = target_corr[abs(target_corr) < 0.10]
print(f"Count: {len(weak)}")
for feat, r in weak.items():
    print(f"  {feat:<32} {r:>+7.3f}")

In [0]:
drop_features = [
    "national_max_temp", "national_min_temp", "national_temp_anomaly",
    "action_intensity", "adaptation_intensity", "adventure_intensity",
    "animated_intensity", "biopic_intensity", "comedy_intensity",
    "crime_intensity", "documentary_intensity", "drama_intensity",
    "family_intensity", "fantasy_intensity", "foreign_intensity",
    "horror_intensity", "musical_intensity", "mystery_intensity",
    "period_intensity", "political_intensity", "romance_intensity",
    "romanticcomedy_intensity", "scifi_intensity", "sequel_intensity",
    "suspense_intensity", "thriller_intensity",
]

baseline_features = [f for f in control_cols if f not in drop_features]
assert len(baseline_features) == 19, f"Expected 19, got {len(baseline_features)}"

n = len(df)
n_test  = 15
n_val   = 15
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].copy()
val_df   = df.iloc[n_train : n_train + n_val].copy()
test_df  = df.iloc[n_train + n_val :].copy()

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train_df[baseline_features])
X_val   = scaler_X.transform(val_df[baseline_features])
X_test  = scaler_X.transform(test_df[baseline_features])

y_train = scaler_y.fit_transform(train_df[[target_col]]).flatten()
y_val   = scaler_y.transform(val_df[[target_col]]).flatten()
y_test  = scaler_y.transform(test_df[[target_col]]).flatten()

print(f"Features    : {len(baseline_features)}")
print(f"Train       : {X_train.shape}  ({train_df['week_date'].min().date()} → {train_df['week_date'].max().date()})")
print(f"Val         : {X_val.shape}  ({val_df['week_date'].min().date()} → {val_df['week_date'].max().date()})")
print(f"Test        : {X_test.shape}  ({test_df['week_date'].min().date()} → {test_df['week_date'].max().date()})")
print(f"Ratio       : {len(baseline_features) / len(train_df):.2f}")

# MMM - Baseline Model

In [0]:
with pm.Model() as baseline_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)

    mu = intercept + pm.math.dot(X_train, beta)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    trace = pm.sample(
        draws=2000, tune=2000,
        target_accept=0.9,
        chains=4, cores=4,
        random_seed=42,
        return_inferencedata=True,
    )

print(f"\nPosterior beta shape: {trace.posterior['beta'].shape}")

summary = az.summary(trace, var_names=["intercept", "beta", "sigma"])
summary["r_hat"] = pd.to_numeric(summary["r_hat"], errors="coerce")

max_rhat    = float(summary["r_hat"].max())
min_ess     = int(summary["ess_bulk"].min())
divergences = int(trace.sample_stats.diverging.sum().item())

print(f"Max R-hat   : {max_rhat:.4f}   (target < 1.01)")
print(f"Min ESS     : {min_ess}        (target > 400)")
print(f"Divergences : {divergences}    (target 0)")

In [0]:
# --- Train predictions ---
with baseline_model:
    pp_train = pm.sample_posterior_predictive(trace, random_seed=42, progressbar=False)

pred_train_s  = pp_train.posterior_predictive["y_obs"].mean(dim=["chain","draw"]).values
pred_train_lo = pp_train.posterior_predictive["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_train_hi = pp_train.posterior_predictive["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_train      = scaler_y.inverse_transform(pred_train_s.reshape(-1,1)).flatten()
pred_train_lo_u = scaler_y.inverse_transform(pred_train_lo.reshape(-1,1)).flatten()
pred_train_hi_u = scaler_y.inverse_transform(pred_train_hi.reshape(-1,1)).flatten()
actual_train    = train_df[target_col].values

mape_train     = mean_absolute_percentage_error(actual_train, pred_train) * 100
r2_train       = r2_score(actual_train, pred_train)
coverage_train = np.mean((actual_train >= pred_train_lo_u) & (actual_train <= pred_train_hi_u)) * 100

# --- Val predictions ---
with pm.Model() as val_model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta      = pm.Normal("beta", mu=0, sigma=1, shape=X_val.shape[1])
    sigma     = pm.HalfNormal("sigma", sigma=1)
    mu_v      = intercept + pm.math.dot(X_val, beta)
    y_obs_v   = pm.Normal("y_obs", mu=mu_v, sigma=sigma, observed=y_val)
    pp_val    = pm.sample_posterior_predictive(
        trace, var_names=["y_obs"], predictions=True,
        random_seed=42, progressbar=False,
    )

pred_val_s  = pp_val.predictions["y_obs"].mean(dim=["chain","draw"]).values
pred_val_lo = pp_val.predictions["y_obs"].quantile(0.025, dim=["chain","draw"]).values
pred_val_hi = pp_val.predictions["y_obs"].quantile(0.975, dim=["chain","draw"]).values

pred_val      = scaler_y.inverse_transform(pred_val_s.reshape(-1,1)).flatten()
pred_val_lo_u = scaler_y.inverse_transform(pred_val_lo.reshape(-1,1)).flatten()
pred_val_hi_u = scaler_y.inverse_transform(pred_val_hi.reshape(-1,1)).flatten()
actual_val    = val_df[target_col].values

mape_val     = mean_absolute_percentage_error(actual_val, pred_val) * 100
r2_val       = r2_score(actual_val, pred_val)
coverage_val = np.mean((actual_val >= pred_val_lo_u) & (actual_val <= pred_val_hi_u)) * 100

print(f"TRAIN  MAPE: {mape_train:.2f}%   R²: {r2_train:.3f}   Coverage: {coverage_train:.1f}%")
print(f"VAL    MAPE: {mape_val:.2f}%   R²: {r2_val:.3f}   Coverage: {coverage_val:.1f}%")

In [0]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.plot(train_df["week_date"], actual_train, "k-", label="Actual", linewidth=1.2)
ax.plot(train_df["week_date"], pred_train, color="steelblue", label="Predicted", linewidth=1.2)
ax.fill_between(train_df["week_date"], pred_train_lo_u, pred_train_hi_u,
                alpha=0.15, color="steelblue")
ax.set_title(f"Train: MAPE {mape_train:.1f}% | R² {r2_train:.2f}")
ax.set_ylabel("Weekly attendance")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(val_df["week_date"], actual_val, "k-o", label="Actual", linewidth=1.5, markersize=4)
ax.plot(val_df["week_date"], pred_val, color="coral", marker="s", label="Predicted", linewidth=1.5, markersize=4)
ax.fill_between(val_df["week_date"], pred_val_lo_u, pred_val_hi_u, alpha=0.15, color="coral")
ax.set_title(f"Val: MAPE {mape_val:.1f}% | R² {r2_val:.2f}")
ax.set_ylabel("Weekly attendance")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Save artifacts ---
# import os, pickle
# save_dir = "/dbfs/FileStore/mmm"
# os.makedirs(save_dir, exist_ok=True)

# trace.to_netcdf(f"{save_dir}/baseline_trace_19feats.nc")

# with open(f"{save_dir}/baseline_artifacts.pkl", "wb") as f:
#     pickle.dump({
#         "baseline_features": baseline_features,
#         "scaler_X": scaler_X,
#         "scaler_y": scaler_y,
#         "control_cols": control_cols,
#         "target_col": target_col,
#         "train_idx": (0, n_train),
#         "val_idx":   (n_train, n_train + n_val),
#         "test_idx":  (n_train + n_val, n),
#         "metrics": {
#             "train_mape": mape_train, "train_r2": r2_train,
#             "val_mape": mape_val,     "val_r2": r2_val,
#         },
#     }, f)

# print(f"\n✓ Trace saved to {save_dir}/baseline_trace_19feats.nc")
# print(f"✓ Artifacts saved to {save_dir}/baseline_artifacts.pkl")

In [0]:
# --- Residuals ---
resid_train = actual_train - pred_train
resid_val   = actual_val - pred_val

# Pct residuals (for scale-free view)
pct_resid_train = resid_train / actual_train * 100
pct_resid_val   = resid_val / actual_val * 100

# --- Overfitting signal: train-val gap ---
print("═" * 60)
print("OVERFITTING CHECK")
print("═" * 60)
print(f"Train MAPE     : {mape_train:>6.2f}%")
print(f"Val   MAPE     : {mape_val:>6.2f}%")
print(f"  Gap          : {mape_val - mape_train:>+6.2f} pts   (>15 pts = concerning)")
print()
print(f"Train R²       : {r2_train:>6.3f}")
print(f"Val   R²       : {r2_val:>6.3f}")
print(f"  Gap          : {r2_train - r2_val:>+6.3f}        (>0.5 = concerning)")
print()

# Residual stats
print("═" * 60)
print("RESIDUAL DISTRIBUTION")
print("═" * 60)
print(f"{'':20} {'Train':>15} {'Val':>15}")
print(f"{'Mean':<20} {resid_train.mean():>15,.0f} {resid_val.mean():>15,.0f}")
print(f"{'Std':<20} {resid_train.std():>15,.0f} {resid_val.std():>15,.0f}")
print(f"{'Min':<20} {resid_train.min():>15,.0f} {resid_val.min():>15,.0f}")
print(f"{'Max':<20} {resid_train.max():>15,.0f} {resid_val.max():>15,.0f}")
print(f"{'Median abs':<20} {np.median(np.abs(resid_train)):>15,.0f} {np.median(np.abs(resid_val)):>15,.0f}")

# Autocorrelation in residuals — pattern that model missed
from scipy.stats import pearsonr
acf_lag1_train, _ = pearsonr(resid_train[:-1], resid_train[1:])
print(f"\nAutocorrelation lag-1 (train residuals): {acf_lag1_train:+.3f}")
print("  Target: near 0. Strong autocorr = model missed temporal structure.")

# Normality of residuals (visual + numerical)
from scipy.stats import shapiro
sw_stat, sw_p = shapiro(resid_train)
print(f"\nShapiro-Wilk normality test on train residuals: p = {sw_p:.4f}")
print("  Target: p > 0.05 = residuals approx normal.")
print("  p < 0.05 just means non-normal — usually not fatal but informative.")

In [0]:
fig, axes = plt.subplots(3, 2, figsize=(15, 11))

# 1. Train residuals over time
ax = axes[0, 0]
ax.plot(train_df["week_date"], resid_train, color="steelblue", linewidth=1, alpha=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(resid_train.std(), color="red", linewidth=0.5, linestyle="--", label=f"±1σ ({resid_train.std():,.0f})")
ax.axhline(-resid_train.std(), color="red", linewidth=0.5, linestyle="--")
ax.set_title("Train residuals over time")
ax.set_ylabel("Residual (actual − pred)")
ax.legend(); ax.grid(alpha=0.3)

# 2. Val residuals over time
ax = axes[0, 1]
ax.bar(val_df["week_date"], resid_val, color="coral", width=5)
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(resid_train.std(), color="red", linewidth=0.5, linestyle="--", label=f"Train ±1σ")
ax.axhline(-resid_train.std(), color="red", linewidth=0.5, linestyle="--")
ax.set_title("Val residuals (compared to train σ band)")
ax.set_ylabel("Residual")
ax.legend(); ax.grid(alpha=0.3)

# 3. Predicted vs Actual scatter (train)
ax = axes[1, 0]
ax.scatter(actual_train, pred_train, alpha=0.5, s=25, color="steelblue", label="Train")
lims = [min(actual_train.min(), pred_train.min()), max(actual_train.max(), pred_train.max())]
ax.plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title("Train: predicted vs actual")
ax.legend(); ax.grid(alpha=0.3)

# 4. Predicted vs Actual scatter (val)
ax = axes[1, 1]
ax.scatter(actual_val, pred_val, alpha=0.7, s=40, color="coral", label="Val")
ax.plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title("Val: predicted vs actual")
ax.legend(); ax.grid(alpha=0.3)

# 5. Residual histogram (train) — should look normal
ax = axes[2, 0]
ax.hist(resid_train, bins=25, color="steelblue", edgecolor="black", alpha=0.7, density=True)
ax.axvline(0, color="black", linewidth=1)
# Overlay normal curve for comparison
from scipy.stats import norm
x_range = np.linspace(resid_train.min(), resid_train.max(), 100)
ax.plot(x_range, norm.pdf(x_range, resid_train.mean(), resid_train.std()),
        "r-", linewidth=1.5, label="Normal fit")
ax.set_title(f"Train residual distribution (Shapiro p={sw_p:.3f})")
ax.set_xlabel("Residual"); ax.set_ylabel("Density")
ax.legend(); ax.grid(alpha=0.3)

# 6. Residuals vs Predicted (heteroscedasticity check)
ax = axes[2, 1]
ax.scatter(pred_train, resid_train, alpha=0.5, s=25, color="steelblue")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
ax.set_title("Residuals vs predicted (want: random cloud, no funnel)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Historical Data Analysis (Descriptive)

In [0]:
# ============================================================
# BLOCK 1: Media column audit — parse Funnel / LOB / Channel
# ============================================================

# --- Identify media columns ---
# Media columns sit between control_cols and total_weekly_attendance
all_cols      = df.columns.tolist()
target_idx    = all_cols.index("total_weekly_attendance")
media_start   = all_cols.index("Awareness_Cineclub_Digital")  # first media col
media_cols    = all_cols[media_start:target_idx]

print(f"Media columns found : {len(media_cols)}")

# --- Parse Funnel / LOB / Channel ---
# Pattern: Funnel_LOB_Channel
# Channel is always last token (Digital / SEM / Social — no underscores)
# Funnel is always first token
# LOB is everything in between (may contain spaces/hyphens but NOT underscores)

def parse_media_col(col):
    parts   = col.split("_")
    funnel  = parts[0].capitalize()   # normalises 'COnversion' → 'Conversion'
    channel = parts[-1]
    lob     = "_".join(parts[1:-1])
    return funnel, lob, channel

media_meta = pd.DataFrame(
    [parse_media_col(c) + (c,) for c in media_cols],
    columns=["funnel", "lob", "channel", "col_name"]
)

# --- Sanity check: expected funnels / channels ---
print("\nFunnels found    :", sorted(media_meta["funnel"].unique()))
print("Channels found   :", sorted(media_meta["channel"].unique()))
print("LOBs found       :", sorted(media_meta["lob"].unique()))
print("\nColumns per funnel:\n", media_meta["funnel"].value_counts())
print("\nColumns per channel:\n", media_meta["channel"].value_counts())

# --- Nulls & zeros audit (full 165 weeks) ---
spend_df   = df[media_cols].copy()
audit_rows = []

for _, row in media_meta.iterrows():
    col    = row["col_name"]
    series = df[col]
    audit_rows.append({
        "funnel"  : row["funnel"],
        "lob"     : row["lob"],
        "channel" : row["channel"],
        "col_name": col,
        "nulls"   : series.isna().sum(),
        "zeros"   : (series == 0).sum(),
        "pct_zero": round((series == 0).mean() * 100, 1),
        "total_spend": series.sum(),
        "mean_weekly": round(series.mean(), 0),
        "max_weekly" : round(series.max(), 0),
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["funnel", "total_spend"], ascending=[True, False]
)

print("\n--- Full Media Audit ---")
print(audit_df.to_string(index=False))

# Flag any columns with > 80% zeros (sparse / likely inactive)
sparse = audit_df[audit_df["pct_zero"] > 80]
print(f"\nSparse columns (>80% zero): {len(sparse)}")
if len(sparse):
    print(sparse[["col_name", "pct_zero", "total_spend"]].to_string(index=False))

In [0]:
# ============================================================
# BLOCK 2: Descriptive statistics — attendance & spend
# ============================================================

# --- Year labels (full 165 weeks including test) ---
# This is DESCRIPTIVE only — no model decisions from 2026 data
df["year_label"] = df["year"].astype(str)

# --- Attendance descriptives ---
print("=" * 60)
print("ATTENDANCE — Annual Summary")
print("=" * 60)
att_annual = df.groupby("year")["total_weekly_attendance"].agg(
    weeks      = "count",
    total      = "sum",
    mean_weekly= "mean",
    median     = "median",
    std        = "std",
    min_week   = "min",
    max_week   = "max",
).round(0)
print(att_annual.to_string())

# --- Spend descriptives by funnel x year ---
print("\n" + "=" * 60)
print("SPEND BY FUNNEL — Annual Summary (CAD)")
print("=" * 60)

for funnel in ["Awareness", "Consideration", "Conversion"]:
    funnel_cols = media_meta.loc[media_meta["funnel"] == funnel, "col_name"].tolist()
    df[f"spend_{funnel}"] = df[funnel_cols].sum(axis=1)

spend_funnel_annual = df.groupby("year")[[
    "spend_Awareness", "spend_Consideration", "spend_Conversion"
]].sum().round(0)
spend_funnel_annual["total_spend"] = spend_funnel_annual.sum(axis=1)
print(spend_funnel_annual.to_string())

# --- Spend descriptives by LOB x year ---
print("\n" + "=" * 60)
print("SPEND BY LOB — Annual Total (CAD), sorted by 3-year total")
print("=" * 60)

lob_list = media_meta["lob"].unique()
for lob in lob_list:
    lob_cols       = media_meta.loc[media_meta["lob"] == lob, "col_name"].tolist()
    df[f"lob_{lob}"] = df[lob_cols].sum(axis=1)

lob_spend_cols = [f"lob_{l}" for l in lob_list]
lob_annual     = df.groupby("year")[lob_spend_cols].sum().T
lob_annual.index = [c.replace("lob_", "") for c in lob_annual.index]
lob_annual["3yr_total"] = lob_annual.sum(axis=1)
lob_annual = lob_annual.sort_values("3yr_total", ascending=False)
print(lob_annual.round(0).to_string())

# --- Zero-spend week count by year (useful for LOB activation timing) ---
print("\n" + "=" * 60)
print("ZERO-SPEND WEEKS BY FUNNEL, BY YEAR")
print("=" * 60)
for funnel in ["Awareness", "Consideration", "Conversion"]:
    col = f"spend_{funnel}"
    zero_by_year = df[df[col] == 0].groupby("year").size()
    print(f"\n{funnel}:")
    print(zero_by_year.to_string())

In [0]:
# ============================================================
# BLOCK 3: Baseline predictions — train + val only (Option A)
# Test set strictly sealed — no actuals, no predictions
# ============================================================
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# Rebuild df_viz (train + val = 150 weeks) if lost from kernel restart
n_train = n - n_val - n_test
df_viz  = df.iloc[:n_train + n_val].copy().reset_index(drop=True)
df_viz["year"] = df_viz["week_date"].dt.year

# --- viz dataset: train + val only (150 weeks) ---
df_viz = pd.concat([train_df, val_df]).copy().reset_index(drop=True)

X_viz_sc = scaler_X.transform(df_viz[baseline_features].values)

# --- Posterior mean coefficients from existing trace ---
beta_mean      = trace.posterior["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)
intercept_mean = trace.posterior["intercept"].values.ravel().mean()

# --- Predict → inverse transform ---
y_pred_sc_viz = intercept_mean + X_viz_sc @ beta_mean
y_pred_viz    = scaler_y.inverse_transform(
                    y_pred_sc_viz.reshape(-1, 1)
                ).ravel()

# --- Verify metrics reproduce ---
actual_train_v = train_df[target_col].values
actual_val_v   = val_df[target_col].values
pred_train_v   = y_pred_viz[:n_train]
pred_val_v     = y_pred_viz[n_train:]

print(f"Train MAPE : {mean_absolute_percentage_error(actual_train_v, pred_train_v)*100:.2f}%  ← expect ~20.34%")
print(f"Val   MAPE : {mean_absolute_percentage_error(actual_val_v,   pred_val_v)*100:.2f}%  ← expect ~28.29%")
print(f"Train R²   : {r2_score(actual_train_v, pred_train_v):.3f}   ← expect ~0.591")
print(f"Val   R²   : {r2_score(actual_val_v,   pred_val_v):.3f}   ← expect ~-0.324")

# --- df_full: all 165 weeks for spend plotting ---
# Rebuilt from df directly — no dependency on train_df/val_df/test_df
n      = len(df)
n_test = 15
n_val  = 15

df_full = df.copy().reset_index(drop=True)
df_full["year"] = df_full["week_date"].dt.year

# --- Attach to df_viz ---
df_viz["baseline_pred"] = y_pred_viz
df_viz["headroom_raw"]  = df_viz[target_col] - df_viz["baseline_pred"]
df_viz["headroom_pct"]  = df_viz["headroom_raw"] / df_viz[target_col] * 100
df_viz["segment"]       = "train"
df_viz.loc[n_train:, "segment"] = "validation"

# --- Funnel spend totals ---
for funnel in ["Awareness", "Consideration", "Conversion"]:
    fcols = media_meta.loc[media_meta["funnel"] == funnel, "col_name"].tolist()
    df_viz[f"spend_{funnel}"] = df_viz[fcols].sum(axis=1)

df_viz["total_spend"] = df_viz[
    [f"spend_{f}" for f in ["Awareness", "Consideration", "Conversion"]]
].sum(axis=1)

print(f"\ndf_viz shape : {df_viz.shape}")
print(f"Date range   : {df_viz['week_date'].min().date()} → {df_viz['week_date'].max().date()}")
print(f"Weeks        : {len(df_viz)}  ← expect 150")
print(f"Segments     :\n{df_viz['segment'].value_counts()}")

In [0]:
# ============================================================
# BLOCK 4: Headroom summary — train + val only
# ============================================================

headroom_annual = df_viz.groupby("year").agg(
    weeks            = ("week_date",              "count"),
    avg_actual       = (target_col,               "mean"),
    avg_baseline     = ("baseline_pred",           "mean"),
    avg_headroom     = ("headroom_raw",            "mean"),
    avg_headroom_pct = ("headroom_pct",            "mean"),
    weeks_positive   = ("headroom_raw",            lambda x: (x > 0).sum()),
    weeks_negative   = ("headroom_raw",            lambda x: (x < 0).sum()),
).round(0)

print("=" * 70)
print("HEADROOM SUMMARY BY YEAR  (train + val only | 2025 partial to Nov 10)")
print("=" * 70)
print(headroom_annual.to_string())

# --- Annual totals: actual vs baseline ---
yearly_totals = df_viz.groupby("year").agg(
    total_actual   = (target_col,      "sum"),
    total_baseline = ("baseline_pred", "sum"),
    total_headroom = ("headroom_raw",  "sum"),
).round(0)

yearly_totals["headroom_pct_of_actual"] = (
    yearly_totals["total_headroom"] / yearly_totals["total_actual"] * 100
).round(1)

print("\n--- Annual Totals: Actual vs Baseline vs Headroom ---")
print(yearly_totals.to_string())

In [0]:
import datetime
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# ============================================================
# SETUP
# ============================================================

SAVE_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/Data_Exploration/"
RUN_TS    = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_LGREY = "#F2F2F2"
CPX_DGREY = "#4A4A4A"
CPX_WHITE = "#FFFFFF"

year_colors   = {2023: CPX_NAVY, 2024: CPX_RED,  2025: CPX_GOLD}
year_labels   = {2023: "2023",   2024: "2024",   2025: "2025"}
years         = [2023, 2024, 2025]
bar_width     = 0.26
bar_offsets   = {2023: -bar_width, 2024: 0, 2025: bar_width}
funnels       = ["Awareness", "Consideration", "Conversion"]
funnel_colors = {"Awareness": CPX_NAVY, "Consideration": CPX_RED, "Conversion": CPX_GOLD}
channel_colors= {"Digital": CPX_NAVY, "SEM": CPX_RED, "Social": CPX_GOLD}

mpl.rcParams.update({
    "figure.facecolor" : CPX_WHITE, "axes.facecolor"   : CPX_WHITE,
    "axes.edgecolor"   : CPX_DGREY, "axes.labelcolor"  : CPX_DGREY,
    "axes.grid"        : True,      "grid.color"       : CPX_LGREY,
    "grid.linewidth"   : 0.8,       "xtick.color"      : CPX_DGREY,
    "ytick.color"      : CPX_DGREY, "text.color"       : CPX_DGREY,
    "font.family"      : "sans-serif", "font.size"     : 10,
    "legend.framealpha": 0.9,       "legend.edgecolor" : CPX_LGREY,
})

def add_watermark(fig):
    fig.text(0.99, 0.01,
             f"Cineplex Marketing Analytics  |  EXH MMM Phase 2  |  Generated: {RUN_TS}",
             ha="right", va="bottom", fontsize=7,
             color=CPX_DGREY, alpha=0.6, fontstyle="italic")

def save_fig(fig, filename):
    add_watermark(fig)
    fig.savefig(SAVE_PATH + filename, dpi=600, bbox_inches="tight")
    print(f"Saved: {filename}")

def clean_name(s):
    return (s.replace(" ", "_").replace("&", "and")
             .replace("-", "").replace("/", "").replace("__", "_"))

# ============================================================
# BUILD df_viz (150 weeks) AND df_full (165 weeks)
# ============================================================

n       = len(df)
n_test  = 15
n_val   = 15
n_train = n - n_val - n_test

# df_viz: train + val only — attendance lines + baseline
df_viz = df.iloc[:n_train + n_val].copy().reset_index(drop=True)
df_viz["year"] = df_viz["week_date"].dt.year

# df_full: all 165 weeks — spend bars and spend-only charts
df_full = df.copy().reset_index(drop=True)
df_full["year"] = df_full["week_date"].dt.year

# ============================================================
# BASELINE PREDICTIONS ON df_viz
# ============================================================

X_viz_sc       = scaler_X.transform(df_viz[baseline_features].values)
beta_mean      = trace.posterior["beta"].values.reshape(-1, len(baseline_features)).mean(axis=0)
intercept_mean = trace.posterior["intercept"].values.ravel().mean()

y_pred_sc_viz  = intercept_mean + X_viz_sc @ beta_mean
y_pred_viz     = scaler_y.inverse_transform(y_pred_sc_viz.reshape(-1, 1)).ravel()

# Verify
print(f"Train MAPE : {mean_absolute_percentage_error(df_viz[target_col].values[:n_train], y_pred_viz[:n_train])*100:.2f}%  ← expect ~20.34%")
print(f"Val   MAPE : {mean_absolute_percentage_error(df_viz[target_col].values[n_train:], y_pred_viz[n_train:])*100:.2f}%  ← expect ~28.29%")
print(f"Train R²   : {r2_score(df_viz[target_col].values[:n_train], y_pred_viz[:n_train]):.3f}   ← expect ~0.591")
print(f"Val   R²   : {r2_score(df_viz[target_col].values[n_train:], y_pred_viz[n_train:]):.3f}   ← expect ~-0.324")

df_viz["baseline_pred"] = y_pred_viz
df_viz["headroom_raw"]  = df_viz[target_col] - df_viz["baseline_pred"]
df_viz["headroom_pct"]  = df_viz["headroom_raw"] / df_viz[target_col] * 100
df_viz["segment"]       = "train"
df_viz.loc[n_train:, "segment"] = "validation"

# ============================================================
# SPEND COLUMNS ON BOTH df_viz AND df_full
# ============================================================

for funnel in funnels:
    fcols = media_meta.loc[media_meta["funnel"] == funnel, "col_name"].tolist()
    df_viz[f"spend_{funnel}"]  = df_viz[fcols].sum(axis=1)
    df_full[f"spend_{funnel}"] = df_full[fcols].sum(axis=1)

df_viz["total_spend"]  = df_viz[[f"spend_{f}"  for f in funnels]].sum(axis=1)
df_full["total_spend"] = df_full[[f"spend_{f}" for f in funnels]].sum(axis=1)

lob_list = media_meta["lob"].unique().tolist()
for lob in lob_list:
    lob_cols = media_meta.loc[media_meta["lob"] == lob, "col_name"].tolist()
    df_viz[f"lob_{lob}"]  = df_viz[lob_cols].sum(axis=1)
    df_full[f"lob_{lob}"] = df_full[lob_cols].sum(axis=1)

lob_totals_sorted = sorted(lob_list, key=lambda l: df_full[f"lob_{l}"].sum(), reverse=True)

# Funnel share on df_full
for funnel in funnels:
    df_full[f"share_{funnel}"] = (
        df_full[f"spend_{funnel}"] / df_full["total_spend"].replace(0, np.nan)
    ).fillna(0)

print(f"\ndf_viz  : {len(df_viz)} weeks ({df_viz['week_date'].min().date()} → {df_viz['week_date'].max().date()})")
print(f"df_full : {len(df_full)} weeks ({df_full['week_date'].min().date()} → {df_full['week_date'].max().date()})")

# ============================================================
# BLOCK 4: Headroom summary
# ============================================================

headroom_annual = df_viz.groupby("year").agg(
    weeks            = ("week_date",    "count"),
    avg_actual       = (target_col,     "mean"),
    avg_baseline     = ("baseline_pred","mean"),
    avg_headroom     = ("headroom_raw", "mean"),
    avg_headroom_pct = ("headroom_pct", "mean"),
    weeks_positive   = ("headroom_raw", lambda x: (x > 0).sum()),
    weeks_negative   = ("headroom_raw", lambda x: (x < 0).sum()),
).round(0)

yearly_totals = df_viz.groupby("year").agg(
    total_actual   = (target_col,      "sum"),
    total_baseline = ("baseline_pred", "sum"),
    total_headroom = ("headroom_raw",  "sum"),
).round(0)
yearly_totals["headroom_pct_of_actual"] = (
    yearly_totals["total_headroom"] / yearly_totals["total_actual"] * 100
).round(1)

print("\n" + "=" * 70)
print("HEADROOM SUMMARY BY YEAR (train + val | 2025 partial to Nov 10)")
print("=" * 70)
print(headroom_annual.to_string())
print("\n--- Annual Totals ---")
print(yearly_totals.to_string())


In [0]:
import matplotlib.dates as mdates

for funnel in funnels:
    fig, ax_l = plt.subplots(figsize=(18, 6))
    ax_r      = ax_l.twinx()
    spend_col = f"spend_{funnel}"

    for year in years:
        gv = df_viz[df_viz["year"] == year].copy().reset_index(drop=True)
        gf = df_full[df_full["year"] == year].copy().reset_index(drop=True)
        c  = year_colors[year]
        lb = year_labels[year]

        # Use row position (1-indexed) as x — avoids duplicate week_number issue
        wk_viz  = np.arange(1, len(gv) + 1)
        wk_full = np.arange(1, len(gf) + 1)

        ax_l.plot(wk_viz, gv[target_col] / 1e6,
                  color=c, linewidth=2.2, label=f"Actual {lb}")
        ax_l.plot(wk_viz, gv["baseline_pred"] / 1e6,
                  color=c, linewidth=1.2, linestyle="--", alpha=0.7,
                  label=f"Baseline {lb}")
        ax_r.bar(wk_full + bar_offsets[year],
                 gf[spend_col] / 1e3,
                 width=bar_width, alpha=0.4, color=c, label=f"Spend {lb}")

    ax_l.axvspan(31, 46, alpha=0.04, color=CPX_DGREY)
    ax_l.annotate("Val period\n(2025 only)",
                  xy=(31.5, ax_l.get_ylim()[1] * 0.88),
                  fontsize=7.5, color=CPX_DGREY, fontstyle="italic")
    ax_l.set_ylabel("Weekly Attendance (millions)", color=CPX_NAVY, fontsize=10)
    ax_r.set_ylabel("Weekly Spend ($000s CAD)", color=CPX_DGREY, fontsize=10)
    ax_l.set_xlabel("Week of Year", fontsize=10)
    ax_l.set_title(
        f"EXH MMM — {funnel} Funnel: Spend vs Attendance (YoY)\n"
        "Solid = Actual (to Nov 10)  |  Dashed = Baseline  |  Bars = Full Year Spend",
        fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax_l.set_xlim(1, 53)
    ax_l.set_axisbelow(True); ax_r.set_axisbelow(True)
    ax_l.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}M"))
    ax_r.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
    ll, la = ax_l.get_legend_handles_labels()
    rl, ra = ax_r.get_legend_handles_labels()
    ax_l.legend(ll, la, loc="upper left",  fontsize=8, ncol=2, title="Attendance", title_fontsize=8)
    ax_r.legend(rl, ra, loc="upper right", fontsize=8, ncol=1, title="Spend",      title_fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"5R_Funnel_DualAxis_{clean_name(funnel)}.png")
    plt.show()

In [0]:
for lob in lob_totals_sorted:
    fig, ax_l = plt.subplots(figsize=(18, 6))
    ax_r      = ax_l.twinx()
    spend_col = f"lob_{lob}"

    for year in years:
        gv = df_viz[df_viz["year"] == year].copy().reset_index(drop=True)
        gf = df_full[df_full["year"] == year].copy().reset_index(drop=True)
        c  = year_colors[year]
        lb = year_labels[year]

        wk_viz  = np.arange(1, len(gv) + 1)
        wk_full = np.arange(1, len(gf) + 1)

        ax_l.plot(wk_viz, gv[target_col] / 1e6,
                  color=c, linewidth=2.2, label=f"Actual {lb}")
        ax_l.plot(wk_viz, gv["baseline_pred"] / 1e6,
                  color=c, linewidth=1.2, linestyle="--", alpha=0.7,
                  label=f"Baseline {lb}")
        ax_r.bar(wk_full + bar_offsets[year],
                 gf[spend_col] / 1e3,
                 width=bar_width, alpha=0.4, color=c, label=f"Spend {lb}")

    ax_l.axvspan(31, 46, alpha=0.04, color=CPX_DGREY)
    ax_l.annotate("Val period\n(2025 only)",
                  xy=(31.5, ax_l.get_ylim()[1] * 0.88),
                  fontsize=7.5, color=CPX_DGREY, fontstyle="italic")
    ax_l.set_ylabel("Weekly Attendance (millions)", color=CPX_NAVY, fontsize=10)
    ax_r.set_ylabel("Weekly Spend ($000s CAD)", color=CPX_DGREY, fontsize=10)
    ax_l.set_xlabel("Week of Year", fontsize=10)
    ax_l.set_title(
        f"EXH MMM — LOB: {lob} Spend vs Attendance (YoY)\n"
        "Solid = Actual (to Nov 10)  |  Dashed = Baseline  |  Bars = Full Year Spend",
        fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax_l.set_xlim(1, 53)
    ax_l.set_axisbelow(True); ax_r.set_axisbelow(True)
    ax_l.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}M"))
    ax_r.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
    ll, la = ax_l.get_legend_handles_labels()
    rl, ra = ax_r.get_legend_handles_labels()
    ax_l.legend(ll, la, loc="upper left",  fontsize=8, ncol=2, title="Attendance", title_fontsize=8)
    ax_r.legend(rl, ra, loc="upper right", fontsize=8, ncol=1, title="Spend",      title_fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"6R_LOB_DualAxis_{clean_name(lob)}.png")
    plt.show()

In [0]:
fig, ax = plt.subplots(figsize=(18, 5))

for year in years:
    gv = df_viz[df_viz["year"] == year].copy().reset_index(drop=True)
    wk = np.arange(1, len(gv) + 1)

    ax.plot(wk, gv[target_col] / 1e6,
            color=year_colors[year], linewidth=2.2, label=f"Actual {year_labels[year]}")
    ax.plot(wk, gv["baseline_pred"] / 1e6,
            color=year_colors[year], linewidth=1.2, linestyle="--", alpha=0.7,
            label=f"Baseline {year_labels[year]}")

ax.axvspan(31, 46, alpha=0.04, color=CPX_DGREY)
ax.annotate("Val period\n(2025 only)", xy=(31.5, ax.get_ylim()[1] * 0.88),
            fontsize=7.5, color=CPX_DGREY, fontstyle="italic")
ax.set_ylabel("Weekly Attendance (millions)", color=CPX_NAVY, fontsize=10)
ax.set_xlabel("Week of Year", fontsize=10)
ax.set_title("EXH MMM — Actual vs Baseline Attendance (YoY)\n"
             "Solid = Actual  |  Dashed = Baseline (controls only, no media)",
             fontsize=12, fontweight="bold", color=CPX_NAVY)
ax.set_xlim(1, 53)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}M"))
ax.legend(fontsize=8, ncol=2, title="Year", title_fontsize=8)
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, "14_Actual_vs_Baseline_Attendance.png")
plt.show()

In [0]:
# ============================================================
# BLOCK 12: Weekly spend bars by funnel (df_full)
# ============================================================

for funnel in funnels:
    fig, ax = plt.subplots(figsize=(18, 5))

    for year in years:
        gf = df_full[df_full["year"] == year].copy().reset_index(drop=True)
        wk = np.arange(1, len(gf) + 1)

        ax.bar(wk + bar_offsets[year],
               gf[f"spend_{funnel}"] / 1e3,
               width=bar_width, color=year_colors[year],
               alpha=0.85, label=year_labels[year])

    ax.set_ylabel("Weekly Spend ($000s CAD)", color=CPX_NAVY, fontsize=10)
    ax.set_xlabel("Week of Year", fontsize=10)
    ax.set_title(f"EXH — Weekly Spend: {funnel} Funnel (YoY) — Full Year",
                 fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax.set_xlim(1, 53)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
    ax.legend(fontsize=9, title="Year", title_fontsize=9)
    ax.set_axisbelow(True)
    plt.tight_layout()
    save_fig(fig, f"12_Weekly_Spend_{clean_name(funnel)}_Funnel.png")
    plt.show()

# ============================================================
# BLOCK 13: Weekly spend bars by LOB (df_full)
# ============================================================

for lob in lob_totals_sorted:
    fig, ax = plt.subplots(figsize=(18, 5))

    for year in years:
        gf = df_full[df_full["year"] == year].copy().reset_index(drop=True)
        wk = np.arange(1, len(gf) + 1)

        ax.bar(wk + bar_offsets[year],
               gf[f"lob_{lob}"] / 1e3,
               width=bar_width, color=year_colors[year],
               alpha=0.85, label=year_labels[year])

    ax.set_ylabel("Weekly Spend ($000s CAD)", color=CPX_NAVY, fontsize=10)
    ax.set_xlabel("Week of Year", fontsize=10)
    ax.set_title(f"EXH — Weekly Spend: {lob} (YoY) — Full Year",
                 fontsize=12, fontweight="bold", color=CPX_NAVY)
    ax.set_xlim(1, 53)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
    ax.legend(fontsize=9, title="Year", title_fontsize=9)
    ax.set_axisbelow(True)
    plt.tight_layout()
    save_fig(fig, f"13_Weekly_Spend_{clean_name(lob)}.png")
    plt.show()

In [0]:
# ============================================================
# BLOCK 7: Annual spend by funnel (df_full)
# ============================================================

funnel_annual = df_full.groupby("year")[[f"spend_{f}" for f in funnels]].sum() / 1e6

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(funnels))
for i, year in enumerate(years):
    vals = [funnel_annual.loc[year, f"spend_{f}"] for f in funnels]
    bars = ax.bar(x + (i-1)*0.25, vals, width=0.25,
                  color=year_colors[year], alpha=0.85, label=year_labels[year])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"${v:.2f}M", ha="center", va="bottom", fontsize=8, color=CPX_DGREY)
ax.set_xticks(x)
ax.set_xticklabels(funnels, fontsize=11)
ax.set_ylabel("Total Spend (CAD millions)", fontsize=10)
ax.set_title("Annual Spend by Funnel (YoY) — Full Year",
             fontsize=13, fontweight="bold", color=CPX_NAVY, pad=12)
ax.legend(fontsize=9, title="Year", title_fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:.1f}M"))
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, "07_Annual_Spend_by_Funnel.png")
plt.show()

In [0]:
# ============================================================
# BLOCK 9: Annual spend by LOB (df_full)
# ============================================================

lob_annual_spend = {
    lob: {year: df_full.loc[df_full["year"] == year, f"lob_{lob}"].sum() / 1e3
          for year in years}
    for lob in lob_list
}
lob_df = pd.DataFrame(lob_annual_spend).T
lob_df["total"] = lob_df.sum(axis=1)
lob_df = lob_df.sort_values("total", ascending=True)

fig, ax = plt.subplots(figsize=(14, 9))
y = np.arange(len(lob_df))
for i, year in enumerate(years):
    bars = ax.barh(y + (i-1)*0.25, lob_df[year], height=0.25,
                   color=year_colors[year], alpha=0.85, label=year_labels[year])
    for bar, v in zip(bars, lob_df[year]):
        if v > 0:
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f"${v:,.0f}k", va="center", fontsize=7, color=CPX_DGREY)
ax.set_yticks(y)
ax.set_yticklabels(lob_df.index, fontsize=9)
ax.set_xlabel("Total Annual Spend ($000s CAD)", fontsize=10)
ax.set_title("Annual Spend by LOB — All LOBs (YoY) — Full Year",
             fontsize=13, fontweight="bold", color=CPX_NAVY, pad=12)
ax.legend(fontsize=9, title="Year", title_fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, "09_Annual_Spend_by_LOB.png")
plt.show()

In [0]:
# ============================================================
# BLOCK 10: Annual spend by channel (df_full)
# ============================================================

channels = ["Digital", "SEM", "Social"]
ch_df = pd.DataFrame({
    ch: df_full.groupby("year")[
        media_meta.loc[media_meta["channel"] == ch, "col_name"].tolist()
    ].sum().sum(axis=1) / 1e6
    for ch in channels
})

# Filter to 2023-2025 only — df_full includes Jan-Feb 2026
ch_df = ch_df.loc[years]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(years))
for i, ch in enumerate(channels):
    vals = ch_df[ch].values
    bars = ax.bar(x + (i-1)*0.25, vals, width=0.25,
                  color=channel_colors[ch], alpha=0.85, label=ch)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"${v:.2f}M", ha="center", va="bottom", fontsize=8, color=CPX_DGREY)
ax.set_xticks(x)
ax.set_xticklabels([year_labels[y] for y in years], fontsize=10)
ax.set_ylabel("Total Spend (CAD millions)", fontsize=10)
ax.set_title("Annual Spend by Channel (YoY) — Full Year",
             fontsize=13, fontweight="bold", color=CPX_NAVY, pad=12)
ax.legend(fontsize=9, title="Channel", title_fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:.1f}M"))
ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, "10_Annual_Spend_by_Channel.png")
plt.show()

In [0]:
# ============================================================
# BLOCK 5R-TOTAL: Total spend (all funnels) dual-axis
# ============================================================

fig, ax_l = plt.subplots(figsize=(18, 6))
ax_r      = ax_l.twinx()

for year in years:
    gv = df_viz[df_viz["year"] == year].copy().reset_index(drop=True)
    gf = df_full[df_full["year"] == year].copy().reset_index(drop=True)
    c  = year_colors[year]
    lb = year_labels[year]

    wk_viz  = np.arange(1, len(gv) + 1)
    wk_full = np.arange(1, len(gf) + 1)

    ax_l.plot(wk_viz, gv[target_col] / 1e6,
              color=c, linewidth=2.2, label=f"Actual {lb}")
    ax_l.plot(wk_viz, gv["baseline_pred"] / 1e6,
              color=c, linewidth=1.2, linestyle="--", alpha=0.7,
              label=f"Baseline {lb}")
    ax_r.bar(wk_full + bar_offsets[year],
             gf["total_spend"] / 1e3,
             width=bar_width, alpha=0.4, color=c, label=f"Spend {lb}")

ax_l.axvspan(31, 46, alpha=0.04, color=CPX_DGREY)
ax_l.annotate("Val period\n(2025 only)",
              xy=(31.5, ax_l.get_ylim()[1] * 0.88),
              fontsize=7.5, color=CPX_DGREY, fontstyle="italic")
ax_l.set_ylabel("Weekly Attendance (millions)", color=CPX_NAVY, fontsize=10)
ax_r.set_ylabel("Total Weekly Spend ($000s CAD)", color=CPX_DGREY, fontsize=10)
ax_l.set_xlabel("Week of Year", fontsize=10)
ax_l.set_title(
    "EXH MMM — Total Spend vs Attendance (YoY)\n"
    "Solid = Actual (to Nov 10)  |  Dashed = Baseline  |  Bars = Total Spend (All Funnels)",
    fontsize=12, fontweight="bold", color=CPX_NAVY)
ax_l.set_xlim(1, 53)
ax_l.set_axisbelow(True); ax_r.set_axisbelow(True)
ax_l.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}M"))
ax_r.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
ll, la = ax_l.get_legend_handles_labels()
rl, ra = ax_r.get_legend_handles_labels()
ax_l.legend(ll, la, loc="upper left",  fontsize=8, ncol=2, title="Attendance", title_fontsize=8)
ax_r.legend(rl, ra, loc="upper right", fontsize=8, ncol=1, title="Spend",      title_fontsize=8)
plt.tight_layout()
save_fig(fig, "5R_Total_Spend_DualAxis.png")
plt.show()